<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V6_AD_Only_Mannheim_Hourly_New_AnomalyD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Zugangsberechtigung auf Drive

from google.colab import drive
drive.mount('/content/drive')

# Gezippte Daten werden entpackt und in hohes Verzeichnis gelegt

!cp "/content/drive/MyDrive/BA_Colab/data.zip" "/content/data.zip"

!unzip -q "/content/data.zip" -d "/content"

!rm "/content/data.zip"
!rm "/content/_MACOSX"


Mounted at /content/drive
rm: cannot remove '/content/_MACOSX': No such file or directory


In [10]:
# ══════════════════════════════════════════════════════════════
# 0 — Setup & Imports
# ══════════════════════════════════════════════════════════════

import os, glob, re, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from shapely import wkb

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve, auc, f1_score, roc_auc_score,
    classification_report, roc_curve
)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
  GPU: NVIDIA L4
  VRAM: 23.7 GB


In [18]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Konfiguration (GEÄNDERT)
# ══════════════════════════════════════════════════════════════

# ── Pfade (anpassen!) ──
DEMAND_BASE        = '/content/data/demand'
STATION_NAMES_PATH = '/content/data/station_names/station_names.parquet'
GEO_INFO_PATH      = '/content/data/geo_information/geo_information.parquet'
WEATHER_PATH       = '/content/data/weather/weather.parquet'
HOLIDAYS_PATH      = '/content/data/holidays/holidays.parquet'
VACATIONS_PATH     = '/content/data/vacations/vacations.parquet'

OUTPUT_PATH        = '/content/data/V6_AD_Mannheim.parquet'

# ── Konfiguration ──
CITY               = 'Mannheim'
NETWORK_NAME       = 'Mannheim'
WEATHER_STATION_ID = 292348
FEDERAL_STATE      = 'Baden-Württemberg'

# ── AD-Hyperparameter ──
WINDOW_SIZE        = 24                   # Stunden (1 Tag) — zurück zu hourly
LATENT_DIM         = 32

# ── Labeling-Schwellen ──
Z_TRAIN_THRESHOLD   = 2.0
Z_ANOMALY_THRESHOLD = 3.0
MIN_EVENTS_PER_DAY  = 3.0
MIN_HIST_MEAN       = 2.0
MIN_ABSOLUTE         = 5

# ── Training ──
FILTER_ZERO_HOURS  = True
SCORE_LAST_N_STEPS = 3
BATCH_SIZE         = 4096
EPOCHS             = 35
LEARNING_RATE      = 1e-3
TRAIN_RATIO        = 0.67
VAL_RATIO          = 0.83

# ══════════════════════════════════════════════════════════════
# [DESIGN-ENTSCHEIDUNG] Feature-Set mit Rolling-Kontext
#
# PROBLEM (V4/V4.1):
#   Der AE sieht nur raw demand. 50 Ausleihen können normal oder anomal
#   sein — abhängig vom Stationskontext. Ohne Kontext kann der AE
#   kontextuelle Anomalien nicht erkennen. AUC-ROC ~0.91, AUC-PR ~0.03.
#
# LÖSUNG:
#   Per-Station Rolling-Features geben dem AE den lokalen Kontext:
#   - demand_ratio_7d:  "Wie viel mehr/weniger als die letzten 7 Tage?"
#   - demand_ratio_24h: "Wie viel mehr/weniger als gestern gleiche Zeit?"
#   - volatility_7d:    "Wie volatil war diese Station zuletzt?"
#
# KEIN LEAKAGE:
#   - Alle Rolling-Features nutzen shift(1) → nur vergangene Daten
#   - Kein hist_mean/hist_std aus dem gesamten Datensatz
#   - z_score wird NUR für Labels verwendet, NICHT als Feature
#
# TRANSFERIERBAR:
#   - Ratios sind stadtunabhängig (2.0 = "doppelt so viel" überall)
#   - Rolling-Stats passen sich automatisch an jede Station an
#   - StandardScaler wird in Target City neu gefittet
#
# LITERATUR-BEGRÜNDUNG:
#   Kontextuelle AD erfordert, dass das Modell den lokalen Kontext kennt.
#   Rolling-Normalization ist ein etablierter Ansatz in der Time-Series-AD
#   Literatur (vgl. Springer: "Contextual anomaly detection on time series",
#   2021; ACM Computing Surveys: "Deep Learning for Time Series AD", 2024).
# ══════════════════════════════════════════════════════════════

FEATURE_COLS = [
    # ── Nachfrage roh ──
    'total_demand', 'n_lends', 'n_returns',
    # ── Temporale Lags (roh) ──
    'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h',
    # ── KONTEXTUELLE Features (Rolling, kein Leakage) ──
    'demand_ratio_7d',                    # total_demand / rolling_mean_7d → relativer Demand
    'demand_ratio_24h',                   # total_demand / rolling_mean_24h → vs. gestern
    'volatility_7d',                      # rolling_std_7d → wie volatil ist die Station?
    # ── Kalender (zyklisch) ──
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos',
    # ── Kalender (binär) ──
    'is_weekend', 'is_holiday', 'is_vacation',
    # ── Wetter ──
    'temperature', 'humidity', 'precipitation', 'wind_speed',
]

# Demand + Kontext Features für Scoring (indices 0-8)
DEMAND_FEATURE_INDICES = list(range(9))

N_FEATURES     = len(FEATURE_COLS)
INPUT_DIM_FLAT = WINDOW_SIZE * N_FEATURES

print(f'═══ V6 Konfiguration (Hourly + Rolling-Kontext) ═══')
print(f'Features:     {N_FEATURES}')
print(f'Demand+Kontext für Scoring: {[FEATURE_COLS[i] for i in DEMAND_FEATURE_INDICES]}')
print(f'Input dim flat: {INPUT_DIM_FLAT}')

═══ V6 Konfiguration (Hourly + Rolling-Kontext) ═══
Features:     22
Demand+Kontext für Scoring: ['total_demand', 'n_lends', 'n_returns', 'demand_lag_1h', 'demand_lag_24h', 'demand_lag_168h', 'demand_ratio_7d', 'demand_ratio_24h', 'volatility_7d']
Input dim flat: 528


In [12]:
# ══════════════════════════════════════════════════════════════
# 2 — Hilfsdaten laden (Stationen, Wetter, Feiertage)
# ══════════════════════════════════════════════════════════════

# ── Station Names & Typ-Klassifikation ──
def classify_station(name: str) -> str:
    if not isinstance(name, str) or name.strip() == '':
        return 'unknown'
    n = name.strip()
    if re.search(r'(?i)^recording[_\-\s]', n):  return 'recording'
    if re.match(r'(?i)^bike[-_]?\s*\d*', n):    return 'bike'
    if re.search(r'(?i)(virtuell|virtual)', n):  return 'virtual'
    if re.fullmatch(r'[\d\s\-_\.#/]+', n):      return 'only_nums'
    return 'real'

station_names = pd.read_parquet(STATION_NAMES_PATH)
station_names = station_names.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
station_names['station_type'] = station_names['station_name'].apply(classify_station)
type_lookup = station_names.set_index('station_name_id')['station_type'].to_dict()
print(f'Station-Types: {station_names["station_type"].value_counts().to_dict()}')

# ── Wetter laden & auf Stunde aggregieren ──
weather = pd.read_parquet(WEATHER_PATH)
weather['timestamp'] = pd.to_datetime(weather['timestamp'], utc=True)
weather_ma = weather[weather['location_id'] == WEATHER_STATION_ID].copy()
weather_ma['hour_ts'] = weather_ma['timestamp'].dt.floor('h')

weather_hourly = (
    weather_ma.groupby('hour_ts')
    .agg(
        temperature   = ('temperature', 'mean'),
        humidity      = ('humidity', 'mean'),
        precipitation = ('precipitation', 'sum'),
        wind_speed    = ('wind_speed', 'mean'),
    )
    .reset_index()
)
print(f'Wetter stündlich: {len(weather_hourly):,} Zeilen | '
      f'{weather_hourly["hour_ts"].min().date()} – {weather_hourly["hour_ts"].max().date()}')

# ── Feiertage & Ferien (nur BaWü) ──
holidays  = pd.read_parquet(HOLIDAYS_PATH)
vacations = pd.read_parquet(VACATIONS_PATH)
holidays['start_date']  = pd.to_datetime(holidays['start_date'])
holidays['end_date']    = pd.to_datetime(holidays['end_date'])
vacations['start_date'] = pd.to_datetime(vacations['start_date'])
vacations['end_date']   = pd.to_datetime(vacations['end_date'])

holidays_bw  = holidays[holidays['federal_state_name'] == FEDERAL_STATE]
vacations_bw = vacations[vacations['federal_state_name'] == FEDERAL_STATE]

holiday_dates = set()
for _, row in holidays_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        holiday_dates.add(d.date())

vacation_dates = set()
for _, row in vacations_bw.iterrows():
    for d in pd.date_range(row['start_date'], row['end_date']):
        vacation_dates.add(d.date())

print(f'Feiertage BaWü: {len(holiday_dates)} Tage | Ferien BaWü: {len(vacation_dates)} Tage')


# ══════════════════════════════════════════════════════════════
# 3 — Demand laden & auf Stunde aggregieren
# ══════════════════════════════════════════════════════════════

files = glob.glob(os.path.join(DEMAND_BASE, CITY, '**', '*.parquet'), recursive=True)
if not files:
    files = glob.glob(os.path.join(DEMAND_BASE, CITY, '*.parquet'))
print(f'Parquet-Dateien gefunden: {len(files)}')

cols = ['network_name', 'timestamp', 'station_id', 'station_name_id',
        'location_id', 'n_lends', 'n_returns']
demand_raw = pd.concat([pd.read_parquet(f, columns=cols) for f in files], ignore_index=True)
demand_raw['timestamp'] = pd.to_datetime(demand_raw['timestamp'], utc=True)

demand_raw['station_type'] = demand_raw['station_name_id'].map(type_lookup).fillna('unknown')
demand = demand_raw[demand_raw['station_type'] == 'real'].copy()

print(f'Demand roh:      {len(demand_raw):,} Zeilen')
print(f'Demand (real):   {len(demand):,} Zeilen')
print(f'Stationen:       {demand["station_id"].nunique()}')
print(f'Zeitraum:        {demand["timestamp"].min().date()} – {demand["timestamp"].max().date()}')

# ── Stündliche Aggregation pro Station ──
demand['hour_ts'] = demand['timestamp'].dt.floor('h')
hourly = (
    demand
    .groupby(['station_id', 'station_name_id', 'location_id', 'hour_ts'])
    .agg(n_lends=('n_lends', 'sum'), n_returns=('n_returns', 'sum'))
    .reset_index()
)
hourly['total_demand'] = hourly['n_lends'] + hourly['n_returns']

# ── Deduplizierung ──
hourly_agg = (
    hourly.groupby(['station_id', 'hour_ts'])
    .agg(
        n_lends         = ('n_lends', 'sum'),
        n_returns       = ('n_returns', 'sum'),
        total_demand    = ('total_demand', 'sum'),
        station_name_id = ('station_name_id', 'first'),
        location_id     = ('location_id', 'first'),
    )
    .reset_index()
)

# ── Lücken füllen ──
all_hours = pd.date_range(
    hourly['hour_ts'].min(), hourly['hour_ts'].max(), freq='h', tz='UTC'
)

station_info = (
    hourly_agg.groupby('station_id')
    .agg(station_name_id=('station_name_id', 'first'), location_id=('location_id', 'first'))
    .reset_index()
)

full_index = pd.MultiIndex.from_product(
    [station_info['station_id'].values, all_hours],
    names=['station_id', 'hour_ts']
)
hourly_full = (
    hourly_agg[['station_id', 'hour_ts', 'n_lends', 'n_returns', 'total_demand']]
    .set_index(['station_id', 'hour_ts'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
hourly_full = hourly_full.merge(station_info, on='station_id', how='left')

print(f'Nach Lückenfüllung: {len(hourly_full):,} Zeilen')
print(f'Stationen: {hourly_full["station_id"].nunique()}')

# ── Stationsfilter ──
n_days = (all_hours[-1] - all_hours[0]).days + 1
station_activity = hourly_full.groupby('station_id')['total_demand'].sum()
min_events = int(n_days * MIN_EVENTS_PER_DAY)
active_stations = station_activity[station_activity >= min_events].index

hourly_full = hourly_full[hourly_full['station_id'].isin(active_stations)].copy()
print(f'Stationen nach Aktivitätsfilter (≥{MIN_EVENTS_PER_DAY}/Tag): '
      f'{hourly_full["station_id"].nunique()} '
      f'(entfernt: {len(station_activity) - len(active_stations)})')

Station-Types: {'bike': 44747, 'real': 13040, 'virtual': 4827, 'recording': 1972, 'only_nums': 51}
Wetter stündlich: 24,913 Zeilen | 2023-04-01 – 2026-02-02
Feiertage BaWü: 167 Tage | Ferien BaWü: 277 Tage
Parquet-Dateien gefunden: 36
Demand roh:      2,683,584 Zeilen
Demand (real):   2,579,329 Zeilen
Stationen:       128
Zeitraum:        2023-03-31 – 2026-02-02
Nach Lückenfüllung: 3,191,936 Zeilen
Stationen: 128
Stationen nach Aktivitätsfilter (≥3.0/Tag): 92 (entfernt: 36)


In [13]:
# ══════════════════════════════════════════════════════════════
# CELL 4 — Feature Engineering (GEÄNDERT — Rolling-Kontext hinzu)
# ══════════════════════════════════════════════════════════════

# ── Kalender-Features (identisch zu V4.1) ──
hourly_full['hour_of_day'] = hourly_full['hour_ts'].dt.hour
hourly_full['day_of_week'] = hourly_full['hour_ts'].dt.dayofweek
hourly_full['month']       = hourly_full['hour_ts'].dt.month
hourly_full['is_weekend']  = (hourly_full['day_of_week'] >= 5).astype(np.int8)
hourly_full['is_holiday']  = hourly_full['hour_ts'].dt.date.apply(
    lambda x: 1 if x in holiday_dates else 0
).astype(np.int8)
hourly_full['is_vacation'] = hourly_full['hour_ts'].dt.date.apply(
    lambda x: 1 if x in vacation_dates else 0
).astype(np.int8)

hourly_full['hour_sin']  = np.sin(2 * np.pi * hourly_full['hour_of_day'] / 24)
hourly_full['hour_cos']  = np.cos(2 * np.pi * hourly_full['hour_of_day'] / 24)
hourly_full['dow_sin']   = np.sin(2 * np.pi * hourly_full['day_of_week'] / 7)
hourly_full['dow_cos']   = np.cos(2 * np.pi * hourly_full['day_of_week'] / 7)
hourly_full['month_sin'] = np.sin(2 * np.pi * hourly_full['month'] / 12)
hourly_full['month_cos'] = np.cos(2 * np.pi * hourly_full['month'] / 12)
print(f'Kalender-Features hinzugefügt.')

# ── Wetter-Features (identisch zu V4.1) ──
hourly_full = hourly_full.merge(weather_hourly, on='hour_ts', how='left')
for col in ['temperature', 'humidity', 'precipitation', 'wind_speed']:
    hourly_full[col] = (
        hourly_full.groupby('station_id')[col]
        .transform(lambda x: x.interpolate(method='linear', limit=6).ffill().bfill())
    )
    median_val = hourly_full[col].median()
    hourly_full[col] = hourly_full[col].fillna(median_val)

print(f'Wetter-Coverage: {hourly_full["temperature"].notna().mean()*100:.1f}%')

# ── Lag-Features (identisch zu V4.1) ──
hourly_full = hourly_full.sort_values(['station_id', 'hour_ts']).reset_index(drop=True)
for lag_name, lag_hours in [('lag_1h', 1), ('lag_24h', 24), ('lag_168h', 168)]:
    hourly_full[f'demand_{lag_name}'] = (
        hourly_full.groupby('station_id')['total_demand'].shift(lag_hours)
    )

# ══════════════════════════════════════════════════════════════
# NEU: Per-Station Rolling-Features (kontextuell, kein Leakage)
#
# shift(1) stellt sicher: NUR vergangene Daten fließen ein.
# Das Modell sieht: "Wie war die Station in den letzten N Stunden?"
# und kann dann beurteilen ob der aktuelle Wert abweicht.
#
# rolling_mean_7d  = Mittelwert der letzten 168h (7 Tage), shifted
# rolling_mean_24h = Mittelwert der letzten 24h, shifted
# rolling_std_7d   = Standardabweichung der letzten 168h, shifted
# ══════════════════════════════════════════════════════════════

print('Rolling-Features berechnen (kann etwas dauern)...')

# 7-Tage Rolling (168h), shifted um 1 → kein Leakage
hourly_full['rolling_mean_7d'] = (
    hourly_full.groupby('station_id')['total_demand']
    .transform(lambda x: x.shift(1).rolling(168, min_periods=24).mean())
)
hourly_full['rolling_std_7d'] = (
    hourly_full.groupby('station_id')['total_demand']
    .transform(lambda x: x.shift(1).rolling(168, min_periods=24).std())
)

# 24h Rolling, shifted um 1 → kein Leakage
hourly_full['rolling_mean_24h'] = (
    hourly_full.groupby('station_id')['total_demand']
    .transform(lambda x: x.shift(1).rolling(24, min_periods=6).mean())
)

# ── Kontextuelle Ratio-Features ──
# demand_ratio = aktueller Demand / kürzlicher Durchschnitt
# → 1.0 = wie erwartet, 2.0 = doppelt so viel, 0.0 = nichts obwohl was erwartet
# Clip auf 0.01 um Division durch Null zu vermeiden

hourly_full['demand_ratio_7d'] = (
    hourly_full['total_demand'] /
    hourly_full['rolling_mean_7d'].clip(lower=0.01)
)

hourly_full['demand_ratio_24h'] = (
    hourly_full['total_demand'] /
    hourly_full['rolling_mean_24h'].clip(lower=0.01)
)

hourly_full['volatility_7d'] = hourly_full['rolling_std_7d'].fillna(0)

# Clippen gegen Extremwerte (Ratio > 20 = 20× so viel wie üblich → cap)
hourly_full['demand_ratio_7d']  = hourly_full['demand_ratio_7d'].clip(0, 20)
hourly_full['demand_ratio_24h'] = hourly_full['demand_ratio_24h'].clip(0, 20)
hourly_full['volatility_7d']    = hourly_full['volatility_7d'].clip(0,
    hourly_full['volatility_7d'].quantile(0.999))

print(f'Rolling-Features hinzugefügt.')

# ── Anlaufphase entfernen (braucht 168h + 1 für Rolling) ──
before = len(hourly_full)
hourly_full = hourly_full.dropna(
    subset=['demand_lag_168h', 'rolling_mean_7d']
).reset_index(drop=True)
print(f'Anlaufphase: {before - len(hourly_full):,} Zeilen entfernt')

# ── Separierbarkeits-Check ──
print(f'\n═══ Rolling-Feature Statistik ═══')
print(f'demand_ratio_7d:   mean={hourly_full["demand_ratio_7d"].mean():.2f}, '
      f'median={hourly_full["demand_ratio_7d"].median():.2f}, '
      f'std={hourly_full["demand_ratio_7d"].std():.2f}')
print(f'demand_ratio_24h:  mean={hourly_full["demand_ratio_24h"].mean():.2f}, '
      f'median={hourly_full["demand_ratio_24h"].median():.2f}, '
      f'std={hourly_full["demand_ratio_24h"].std():.2f}')
print(f'volatility_7d:     mean={hourly_full["volatility_7d"].mean():.2f}, '
      f'median={hourly_full["volatility_7d"].median():.2f}')



Kalender-Features hinzugefügt.
Wetter-Coverage: 100.0%
Rolling-Features berechnen (kann etwas dauern)...
Rolling-Features hinzugefügt.
Anlaufphase: 15,456 Zeilen entfernt

═══ Rolling-Feature Statistik ═══
demand_ratio_7d:   mean=0.96, median=0.00, std=1.83
demand_ratio_24h:  mean=1.05, median=0.00, std=2.23
volatility_7d:     mean=1.90, median=1.47


In [14]:
# ══════════════════════════════════════════════════════════════
# CELL 5 — Temporaler Split + Labeling (GEÄNDERT — minimal)
#
# Identisch zu V4.1, aber:
#  - Stats IMMER nur aus Trainingsperiode (wie V4.1)
#  - z_Score NUR für Labeling, NICHT in Features
#  - Zusätzlich: Check dass Rolling-Features bei Anomalien anders sind
# ══════════════════════════════════════════════════════════════

t_min = hourly_full['hour_ts'].min()
t_max = hourly_full['hour_ts'].max()
total_hours = (t_max - t_min).total_seconds() / 3600

train_end = t_min + pd.Timedelta(hours=int(total_hours * TRAIN_RATIO))
val_end   = t_min + pd.Timedelta(hours=int(total_hours * VAL_RATIO))

print(f'\n═══ Temporaler Split ═══')
print(f'Train-Ende: {train_end.date()}')
print(f'Val-Ende:   {val_end.date()}')
print(f'Test-Ende:  {t_max.date()}')

# ── Stats NUR aus Trainingsperiode ──
train_period = hourly_full[hourly_full['hour_ts'] < train_end]

stats = (
    train_period
    .groupby(['station_id', 'day_of_week', 'hour_of_day'])['total_demand']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'hist_mean', 'std': 'hist_std'})
)

hourly_full = hourly_full.merge(
    stats, on=['station_id', 'day_of_week', 'hour_of_day'], how='left'
)

global_mean = train_period['total_demand'].mean()
global_std  = train_period['total_demand'].std()
hourly_full['hist_mean'] = hourly_full['hist_mean'].fillna(global_mean)
hourly_full['hist_std']  = hourly_full['hist_std'].fillna(global_std)
hourly_full['hist_std']  = hourly_full['hist_std'].clip(lower=0.1)

hourly_full['z_score'] = (
    (hourly_full['total_demand'] - hourly_full['hist_mean']) / hourly_full['hist_std']
)

# ── Labeling (identisch zu V4.1) ──
hourly_full['label'] = 'normal'

is_anomaly = (
    (hourly_full['z_score'].abs() > Z_ANOMALY_THRESHOLD) &
    (hourly_full['hist_mean'] >= MIN_HIST_MEAN) &
    (hourly_full['total_demand'] >= MIN_ABSOLUTE)
)
is_grauzone = (
    (hourly_full['z_score'].abs() > Z_TRAIN_THRESHOLD) &
    ~is_anomaly
)

hourly_full.loc[is_grauzone, 'label'] = 'grauzone'
hourly_full.loc[is_anomaly, 'label']  = 'anomal'

print('\n═══ Label-Verteilung ═══')
print(hourly_full['label'].value_counts())
print(f'Anomalie-Rate: {is_anomaly.mean():.4f}')

# ── NEU: Separierbarkeits-Check mit Rolling-Features ──
normal_mask = hourly_full['label'] == 'normal'
anomal_mask = hourly_full['label'] == 'anomal'

print(f'\n═══ Rolling-Features: Normal vs. Anomal ═══')
for feat in ['demand_ratio_7d', 'demand_ratio_24h', 'volatility_7d']:
    n_mean = hourly_full.loc[normal_mask, feat].mean()
    a_mean = hourly_full.loc[anomal_mask, feat].mean()
    print(f'  {feat:<22} Normal: {n_mean:.2f}  |  Anomal: {a_mean:.2f}  |  '
          f'Ratio: {a_mean/max(n_mean, 0.01):.1f}×')

print(f'\n  → Wenn Anomal/Normal Ratio >> 1 ist, kann der AE die Kontextfeatures nutzen.')



═══ Temporaler Split ═══
Train-Ende: 2025-02-27
Val-Ende:   2025-08-11
Test-Ende:  2026-02-02

═══ Label-Verteilung ═══
label
normal      2157866
grauzone     113644
anomal         7238
Name: count, dtype: int64
Anomalie-Rate: 0.0032

═══ Rolling-Features: Normal vs. Anomal ═══
  demand_ratio_7d        Normal: 0.76  |  Anomal: 5.25  |  Ratio: 6.9×
  demand_ratio_24h       Normal: 0.84  |  Anomal: 4.90  |  Ratio: 5.8×
  volatility_7d          Normal: 1.88  |  Anomal: 4.29  |  Ratio: 2.3×

  → Wenn Anomal/Normal Ratio >> 1 ist, kann der AE die Kontextfeatures nutzen.


In [7]:
# ══════════════════════════════════════════════════════════════
# CELL 6 — Speichern (GEÄNDERT — Pfad)
# ══════════════════════════════════════════════════════════════

hourly_full['network_name'] = NETWORK_NAME
hourly_full.to_parquet(OUTPUT_PATH, index=False)
print(f'\n✅ Gespeichert: {OUTPUT_PATH}')
print(f'   Shape: {hourly_full.shape}')



✅ Gespeichert: /content/data/V6_AD_Mannheim.parquet
   Shape: (2278748, 37)


In [15]:
# ══════════════════════════════════════════════════════════════
# CELL 7 — Split & Sequenzen (IDENTISCH zu V4.1)
#
# Einziger Unterschied: FEATURE_COLS enthält jetzt die
# Rolling-Features → der Scaler normalisiert sie automatisch mit.
# ══════════════════════════════════════════════════════════════

# Train: NUR normale Daten
df_train = hourly_full[
    (hourly_full['hour_ts'] < train_end) &
    (hourly_full['label'] == 'normal')
].copy()

# Active-Hours Filtering
if FILTER_ZERO_HOURS:
    before_zero = len(df_train)
    df_train = df_train[df_train['total_demand'] > 0].copy()
    print(f'[V6] Zero-Stunden aus Training entfernt: {before_zero - len(df_train):,} '
          f'({(before_zero - len(df_train))/max(before_zero,1):.1%})')

# Val & Test (ohne Grauzone)
df_val = hourly_full[
    (hourly_full['hour_ts'] >= train_end) &
    (hourly_full['hour_ts'] < val_end) &
    (hourly_full['label'] != 'grauzone')
].copy()

df_test = hourly_full[
    (hourly_full['hour_ts'] >= val_end) &
    (hourly_full['label'] != 'grauzone')
].copy()

print(f'\n═══ Split ═══')
print(f'Train (normal, active): {len(df_train):>9,} | bis {train_end.date()}')
print(f'Val   (ohne Grauzone):  {len(df_val):>9,} | '
      f'Anomalien: {(df_val["label"]=="anomal").sum():,} '
      f'({(df_val["label"]=="anomal").mean():.2%})')
print(f'Test  (ohne Grauzone):  {len(df_test):>9,} | '
      f'Anomalien: {(df_test["label"]=="anomal").sum():,} '
      f'({(df_test["label"]=="anomal").mean():.2%})')

# ── Normalisierung ──
scaler = StandardScaler()
train_scaled = scaler.fit_transform(df_train[FEATURE_COLS].values)
val_scaled   = scaler.transform(df_val[FEATURE_COLS].values)
test_scaled  = scaler.transform(df_test[FEATURE_COLS].values)
print(f'Scaler fitted auf {len(train_scaled):,} Trainings-Samples')

# ── Sequenzen ──
def create_station_sequences(df, scaled_data, window_size):
    sequences, labels, meta = [], [], []
    station_ids = df['station_id'].values
    timestamps  = df['hour_ts'].values
    label_arr   = df['label'].values

    for station in df['station_id'].unique():
        mask = station_ids == station
        s_data   = scaled_data[mask]
        s_labels = label_arr[mask]
        s_times  = timestamps[mask]

        if len(s_data) < window_size:
            continue

        for i in range(len(s_data) - window_size + 1):
            sequences.append(s_data[i:i + window_size])
            labels.append(s_labels[i + window_size - 1])
            meta.append({'station_id': station, 'timestamp': s_times[i + window_size - 1]})

    return np.array(sequences, dtype=np.float32), np.array(labels), meta

print(f'\nSequenzen erstellen (Window={WINDOW_SIZE})...')
X_train, y_train, meta_train = create_station_sequences(df_train, train_scaled, WINDOW_SIZE)
X_val,   y_val,   meta_val   = create_station_sequences(df_val,   val_scaled,   WINDOW_SIZE)
X_test,  y_test,  meta_test  = create_station_sequences(df_test,  test_scaled,  WINDOW_SIZE)

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}    (Anomalien: {(y_val=="anomal").sum()})')
print(f'X_test:  {X_test.shape}   (Anomalien: {(y_test=="anomal").sum()})')

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)
print(f'Flattened dim: {INPUT_DIM_FLAT}')

# ── Sanity Checks ──
print(f'\n═══ SANITY CHECKS ═══')
assert 'z_score' not in FEATURE_COLS, "z_score darf NICHT in FEATURE_COLS!"
assert 'demand_residual' not in FEATURE_COLS, "demand_residual darf NICHT in Features!"
assert 'demand_ratio_7d' in FEATURE_COLS, "demand_ratio_7d MUSS in Features sein!"
assert (y_train == 'normal').all(), "y_train enthält nicht-normale Labels!"
assert (y_val == 'anomal').sum() > 0, "Keine Anomalien in Validation!"
assert (y_test == 'anomal').sum() > 0, "Keine Anomalien in Test!"
print('✅ Alle Checks bestanden.\n')


[V6] Zero-Stunden aus Training entfernt: 850,397 (58.4%)

═══ Split ═══
Train (normal, active):   605,310 | bis 2025-02-27
Val   (ohne Grauzone):    338,014 | Anomalien: 1,637 (0.48%)
Test  (ohne Grauzone):    366,999 | Anomalien: 1,217 (0.33%)
Scaler fitted auf 605,310 Trainings-Samples

Sequenzen erstellen (Window=24)...
X_train: (603216, 24, 22)
X_val:   (335898, 24, 22)    (Anomalien: 1633)
X_test:  (364883, 24, 22)   (Anomalien: 1211)
Flattened dim: 528

═══ SANITY CHECKS ═══
✅ Alle Checks bestanden.



In [16]:
# ══════════════════════════════════════════════════════════════
# 9 — Modellarchitekturen
# ══════════════════════════════════════════════════════════════

class VanillaAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, latent_dim), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def get_latent(self, x):
        return self.encoder(x)


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=32, n_layers=2, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim

        self.encoder = nn.LSTM(
            input_size=n_features, hidden_size=latent_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.decoder = nn.LSTM(
            input_size=latent_dim, hidden_size=latent_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        self.output_layer = nn.Linear(latent_dim, n_features)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        _, (hidden, cell) = self.encoder(x)
        latent = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        decoder_out, _ = self.decoder(latent, (hidden, cell))
        return self.output_layer(decoder_out)

    def get_latent(self, x):
        _, (hidden, _) = self.encoder(x)
        return hidden[-1]


class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, 128), nn.ReLU()
        )
        self.fc_mu     = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(256, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    def get_latent(self, x):
        mu, _ = self.encode(x)
        return mu


# ══════════════════════════════════════════════════════════════
# 10 — Training Framework
# ══════════════════════════════════════════════════════════════

def train_ae(model, X_train, X_val, model_name='AE',
             epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE,
             is_vae=False, vae_beta=1.0):

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=7
    )
    criterion = nn.MSELoss()
    scaler_amp = torch.amp.GradScaler('cuda')

    train_tensor = torch.FloatTensor(X_train).to(device)
    train_loader = DataLoader(
        TensorDataset(train_tensor, train_tensor),
        batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=0, pin_memory=False
    )

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    EARLY_STOP_PATIENCE = 15

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, _ in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                if is_vae:
                    x_hat, mu, logvar = model(batch_x)
                    recon_loss = criterion(x_hat, batch_x)
                    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
                    loss = recon_loss + vae_beta * kl_loss
                else:
                    x_hat = model(batch_x)
                    loss = criterion(x_hat, batch_x)

            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
            epoch_loss += loss.item()

        avg_train_loss = epoch_loss / len(train_loader)

        # Validation (alle 3 Epochs)
        if (epoch + 1) % 3 == 0 or epoch == 0 or epoch == epochs - 1:
            model.eval()
            val_losses = []
            with torch.no_grad(), torch.amp.autocast('cuda'):
                for vi in range(0, len(X_val), 4096):
                    v_chunk = torch.FloatTensor(X_val[vi:vi+4096]).to(device)
                    if is_vae:
                        v_hat, v_mu, v_lv = model(v_chunk)
                        v_recon = criterion(v_hat, v_chunk)
                        v_kl = -0.5 * torch.mean(1 + v_lv - v_mu.pow(2) - v_lv.exp())
                        val_losses.append((v_recon + vae_beta * v_kl).item())
                    else:
                        v_hat = model(v_chunk)
                        val_losses.append(criterion(v_hat, v_chunk).item())
                    del v_chunk, v_hat
            val_loss = np.mean(val_losses)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 3
        else:
            val_loss = history['val_loss'][-1] if history['val_loss'] else avg_train_loss

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  [{model_name}] Epoch {epoch+1:>3}/{epochs} | '
                  f'Train: {avg_train_loss:.6f} | Val: {val_loss:.6f} | LR: {lr_now:.1e}')

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f'  [{model_name}] Early stopping at epoch {epoch+1}')
            break

    if best_state:
        model.load_state_dict(best_state)
        model = model.to(device)
    print(f'  [{model_name}] Best Val Loss: {best_val_loss:.6f}\n')
    return history


    # ══════════════════════════════════════════════════════════════
# 11 — Scoring & Evaluation
#
# [FIX #6] Scoring über GANZE Sequenz, nicht nur letzte N Stunden
# BEGRÜNDUNG: "Letzte 3h" war ein Hyperparameter, der auf das eigene
# Labeling-Schema optimiert. Die ganze Sequenz ist konservativer
# und ehrlicher. Der AE-Reconstruction-Error über 24h ist ein
# robusteres Anomalie-Signal.
#
# [FIX #4] Demand-only Scoring: Nur die 6 Demand-Features werden
# für den Score herangezogen. Wetter/Kalender sollen rekonstruiert
# werden (hilft dem AE), aber nicht in den Score einfließen.
# ══════════════════════════════════════════════════════════════

def compute_anomaly_scores(model, X, is_vae=False, chunk_size=2048,
                           demand_indices=DEMAND_FEATURE_INDICES):
    """
    Reconstruction Error NUR auf Demand-Features, über die GANZE Sequenz.
    """
    model.eval()
    model = model.to(device)
    all_scores = []

    for i in range(0, len(X), chunk_size):
        chunk = torch.FloatTensor(X[i:i+chunk_size]).to(device)
        with torch.no_grad(), torch.amp.autocast('cuda'):
            if is_vae:
                x_hat, _, _ = model(chunk)
            else:
                x_hat = model(chunk)

            if chunk.dim() == 3:
                # LSTM: (batch, seq_len, features) → Demand-Features filtern
                diff = (chunk[:, :, demand_indices] - x_hat[:, :, demand_indices]) ** 2
            else:
                # Flat: (batch, seq_len*features) → reshape, filtern
                batch_size = chunk.shape[0]
                c_3d = chunk.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                h_3d = x_hat.reshape(batch_size, WINDOW_SIZE, N_FEATURES)
                diff = (c_3d[:, :, demand_indices] - h_3d[:, :, demand_indices]) ** 2

            # Mean über ganze Sequenz und alle Demand-Features
            scores = diff.mean(dim=tuple(range(1, diff.dim())))

        all_scores.append(scores.cpu())
        del chunk, x_hat, scores, diff
        torch.cuda.empty_cache()

    return torch.cat(all_scores).numpy()


def find_best_threshold(scores, labels):
    """Schwellenwert der den F1-Score auf Val maximiert."""
    binary = (labels == 'anomal').astype(int)
    if binary.sum() == 0:
        return np.percentile(scores, 95), 0.0
    prec, rec, thresholds = precision_recall_curve(binary, scores)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-8)
    best_idx = np.argmax(f1[:-1])
    return thresholds[best_idx], f1[best_idx]


def evaluate_model(model, X, y, threshold, model_name='AE', is_vae=False):
    """Evaluation mit allen Metriken."""
    scores = compute_anomaly_scores(model, X, is_vae=is_vae)
    binary = (y == 'anomal').astype(int)
    preds  = (scores > threshold).astype(int)

    prec, rec, _ = precision_recall_curve(binary, scores)
    auc_pr = auc(rec, prec)
    f1 = f1_score(binary, preds, zero_division=0)

    try:
        auc_roc = roc_auc_score(binary, scores)
    except ValueError:
        auc_roc = 0.0

    print(f'── {model_name} auf Testdaten ──')
    print(f'  AUC-PR:  {auc_pr:.4f}  (Hauptmetrik)')
    print(f'  F1:      {f1:.4f}')
    print(f'  AUC-ROC: {auc_roc:.4f}')
    print(f'  Threshold: {threshold:.6f}')
    print(classification_report(binary, preds, target_names=['Normal', 'Anomal'],
                                zero_division=0))

    return {
        'model': model_name, 'auc_pr': auc_pr, 'f1': f1, 'auc_roc': auc_roc,
        'threshold': threshold, 'scores': scores, 'preds': preds
    }



═══ Split ═══
Train (nur normal):    1,455,707 | bis 2025-02-27
  davon demand=0:        850,397 (58.4%)
Val   (ohne Grauzone):   338,014 | Anomalien: 1,637 (0.48%)
Test  (ohne Grauzone):   366,999 | Anomalien: 1,217 (0.33%)

Scaler fitted auf 1,455,707 Trainings-Samples

Scaler-Means (ausgewählte Features):
  total_demand: mean=1.272, std=2.698
  n_lends: mean=0.635, std=1.520
  n_returns: mean=0.637, std=1.600
  demand_lag_1h: mean=1.442, std=3.049
  demand_lag_24h: mean=1.449, std=3.041
  demand_lag_168h: mean=1.454, std=3.051

Sequenzen erstellen (Window=24)...
X_train: (1453591, 24, 22)  (nur normal)
X_val:   (335898, 24, 22)    (Anomalien: 1633)
X_test:  (364883, 24, 22)   (Anomalien: 1211)
Flattened dim: 528

═══ SANITY CHECKS ═══
y_train: (array(['normal'], dtype='<U6'), array([1453591]))
y_val:   (array(['anomal', 'normal'], dtype='<U6'), array([  1633, 334265]))
y_test:  (array(['anomal', 'normal'], dtype='<U6'), array([  1211, 363672]))
✅ Alle Checks bestanden — keine Leaka

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — Training & Evaluation (UNVERÄNDERT — Dimensionen passen sich an)
# ══════════════════════════════════════════════════════════════

print('='*60)
print('MODELL 1: Vanilla Autoencoder (Baseline)')
print('='*60)
torch.cuda.empty_cache()

vanilla_ae = VanillaAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vanilla_ae.parameters()):,}')

hist_vanilla = train_ae(vanilla_ae, X_train_flat, X_val_flat,
                        model_name='VanillaAE', epochs=EPOCHS, batch_size=BATCH_SIZE)

scores_val_v = compute_anomaly_scores(vanilla_ae, X_val_flat)
thresh_v, f1_v = find_best_threshold(scores_val_v, y_val)
print(f'Val-Threshold: {thresh_v:.6f} (F1={f1_v:.4f})')
results_vanilla = evaluate_model(vanilla_ae, X_test_flat, y_test, thresh_v, 'VanillaAE')
vanilla_ae = vanilla_ae.cpu()
torch.cuda.empty_cache()
print('✅ Vanilla AE fertig\n')

In [19]:
print('='*60)
print('MODELL 2: LSTM-Autoencoder')
print('='*60)
torch.cuda.empty_cache()

lstm_ae = LSTMAutoencoder(n_features=N_FEATURES, latent_dim=LATENT_DIM, n_layers=2)
print(f'Parameter: {sum(p.numel() for p in lstm_ae.parameters()):,}')

hist_lstm = train_ae(lstm_ae, X_train, X_val,
                     model_name='LSTM-AE', epochs=EPOCHS, batch_size=2048)

scores_val_l = compute_anomaly_scores(lstm_ae, X_val)
thresh_l, f1_l = find_best_threshold(scores_val_l, y_val)
print(f'Val-Threshold: {thresh_l:.6f} (F1={f1_l:.4f})')
results_lstm = evaluate_model(lstm_ae, X_test, y_test, thresh_l, 'LSTM-AE')
lstm_ae = lstm_ae.cpu()
torch.cuda.empty_cache()
print('✅ LSTM-AE fertig\n')

MODELL 2: LSTM-Autoencoder
Parameter: 33,238
  [LSTM-AE] Epoch   1/35 | Train: 0.450463 | Val: 0.343500 | LR: 1.0e-03
  [LSTM-AE] Epoch   5/35 | Train: 0.218114 | Val: 0.281332 | LR: 1.0e-03
  [LSTM-AE] Epoch  10/35 | Train: 0.174985 | Val: 0.218137 | LR: 1.0e-03
  [LSTM-AE] Epoch  15/35 | Train: 0.158630 | Val: 0.200491 | LR: 1.0e-03
  [LSTM-AE] Epoch  20/35 | Train: 0.147489 | Val: 0.191139 | LR: 1.0e-03
  [LSTM-AE] Epoch  25/35 | Train: 0.139684 | Val: 0.179972 | LR: 1.0e-03
  [LSTM-AE] Epoch  30/35 | Train: 0.133758 | Val: 0.170022 | LR: 1.0e-03
  [LSTM-AE] Epoch  35/35 | Train: 0.128391 | Val: 0.165225 | LR: 1.0e-03
  [LSTM-AE] Best Val Loss: 0.165225

Val-Threshold: 0.874676 (F1=0.0588)
── LSTM-AE auf Testdaten ──
  AUC-PR:  0.0296  (Hauptmetrik)
  F1:      0.0576
  AUC-ROC: 0.8989
  Threshold: 0.874676
              precision    recall  f1-score   support

      Normal       1.00      0.96      0.98    363672
      Anomal       0.03      0.39      0.06      1211

    accuracy   

In [ ]:

print('='*60)
print('MODELL 3: Variational Autoencoder (VAE)')
print('='*60)
torch.cuda.empty_cache()

vae = VAE(input_dim=INPUT_DIM_FLAT, latent_dim=LATENT_DIM)
print(f'Parameter: {sum(p.numel() for p in vae.parameters()):,}')

hist_vae = train_ae(vae, X_train_flat, X_val_flat,
                    model_name='VAE', epochs=EPOCHS, batch_size=BATCH_SIZE,
                    is_vae=True, vae_beta=0.5)

scores_val_vae = compute_anomaly_scores(vae, X_val_flat, is_vae=True)
thresh_vae, f1_vae = find_best_threshold(scores_val_vae, y_val)
print(f'Val-Threshold: {thresh_vae:.6f} (F1={f1_vae:.4f})')
results_vae = evaluate_model(vae, X_test_flat, y_test, thresh_vae, 'VAE', is_vae=True)
vae = vae.cpu()
torch.cuda.empty_cache()
print('✅ VAE fertig\n')